In [22]:
import numpy as np
import xarray as xr
from matplotlib import pyplot as plt
import pandas as pd
import glob
import os
# ----------------------------------------------------------------------------
# Paths
# ----------------------------------------------------------------------------
path_ensemble="/p/projects/megarun/luciagu/data/tabone2024/ensemble_reduced/*"
path_output="../output"

sim_paths = sorted(glob.glob(path_ensemble))



In [ ]:
# ---------------------------------------------
# Calculo timeseries: A V
# ---------------------------------------------
V_ensemble = []
A_ensemble = []
A_g_ensemble = []
T_ensemble = []
valid_sim_indices = []   

for i, sim_path in enumerate(sim_paths):

    file_path = os.path.join(sim_path, "yelmo1D.nc")

    try:
        yelmo = xr.open_dataset(file_path)
    except FileNotFoundError:
        print(f"[ERROR] No se encontró el archivo: {file_path}. Se omite esta simulación.")
        continue
    except Exception as e:
        print(f"[ERROR] No se pudo abrir {file_path}: {e}. Se omite esta simulación.")
        continue

    time=yelmo.time.values
    A_ice=yelmo.A_ice.values
    A_ice_g=yelmo.A_ice_g.values
    V_sle=yelmo.V_sle.values
    T=yelmo.T_srf.values
    
    V_ensemble.append(V_sle)
    A_ensemble.append(A_ice)
    A_g_ensemble.append(A_ice_g)
    T_ensemble.append(T)
    valid_sim_indices.append(i)

# convertir lista → array (simulación x tiempo)
V_ensemble = np.stack(V_ensemble, axis=0)
A_ensemble = np.stack(A_ensemble, axis=0)
A_g_ensemble = np.stack(A_g_ensemble, axis=0)
T_ensemble = np.stack(T_ensemble, axis=0)

# construir dataset final
ds_ensemble = xr.Dataset(
    data_vars={
        "A_ice": (("sim", "time"), A_ensemble),
        "A_ice_g": (("sim", "time"), A_g_ensemble),
        "V_sle": (("sim", "time"), V_ensemble),
        "T_srf": (("sim", "time"), T_ensemble)
    },
    coords={
        "sim": np.array(valid_sim_indices),
        "time": time
    }
)

ds_ensemble.to_netcdf("../output/timeseries.nc")

In [24]:
# ---------------------------------------------
# Calculo timeseries: temperature over the ocean
# ---------------------------------------------
T_ann_ensemble = []
T_sum_ensemble = []
valid_sim_indices = []   

for i, sim_path in enumerate(sim_paths):

    file_path1 = os.path.join(sim_path, "yelmo2D_temperatures.nc")
    file_path2 = os.path.join(sim_path, "yelmo2D_reduced.nc")

    try:
        yelmo1 = xr.open_dataset(file_path1)
        yelmo2 = xr.open_dataset(file_path2)
    except FileNotFoundError:
        print(f"[ERROR] No se encontró el archivo: {file_path1}. Se omite esta simulación.")
        continue
    except Exception as e:
        print(f"[ERROR] No se pudo abrir {file_path1}: {e}. Se omite esta simulación.")
        continue

    time=yelmo1.time.values
    T_ann=(yelmo1.Ta_ann.where(yelmo2.mask_bed==0)).mean(dim=["xc","yc"])
    T_sum=(yelmo1.Ta_sum.where(yelmo2.mask_bed==0)).mean(dim=["xc","yc"])
    
    T_ann_ensemble.append(T_ann)
    T_sum_ensemble.append(T_sum)
    valid_sim_indices.append(i)

# convertir lista → array (simulación x tiempo)
T_ann_ensemble = np.stack(T_ann_ensemble, axis=0)
T_sum_ensemble = np.stack(T_sum_ensemble, axis=0)

# construir dataset final
ds_ensemble = xr.Dataset(
    data_vars={
        "T_ann": (("sim", "time"), T_ann_ensemble),
        "T_sum": (("sim", "time"), T_sum_ensemble)
    },
    coords={
        "sim": np.array(valid_sim_indices),
        "time": time
    }
)

ds_ensemble.to_netcdf("../output/timeseries_temperatures.nc")

[ERROR] No se encontró el archivo: /p/projects/megarun/luciagu/data/tabone2024/ensemble_reduced/0/yelmo2D_temperatures.nc. Se omite esta simulación.
[ERROR] No se encontró el archivo: /p/projects/megarun/luciagu/data/tabone2024/ensemble_reduced/100/yelmo2D_temperatures.nc. Se omite esta simulación.
[ERROR] No se encontró el archivo: /p/projects/megarun/luciagu/data/tabone2024/ensemble_reduced/101/yelmo2D_temperatures.nc. Se omite esta simulación.
[ERROR] No se encontró el archivo: /p/projects/megarun/luciagu/data/tabone2024/ensemble_reduced/102/yelmo2D_temperatures.nc. Se omite esta simulación.
[ERROR] No se encontró el archivo: /p/projects/megarun/luciagu/data/tabone2024/ensemble_reduced/103/yelmo2D_temperatures.nc. Se omite esta simulación.
[ERROR] No se encontró el archivo: /p/projects/megarun/luciagu/data/tabone2024/ensemble_reduced/104/yelmo2D_temperatures.nc. Se omite esta simulación.
[ERROR] No se encontró el archivo: /p/projects/megarun/luciagu/data/tabone2024/ensemble_reduced/